# Car Price Prediction using Machine Learning
**CodeAlpha Data Science Internship — Task 3**

This project builds a machine learning model to predict the resale price of used cars based on various features such as present price, kilometers driven, fuel type, and transmission. We explore multiple regression algorithms, compare their performance, and identify the key factors that influence car pricing in the used vehicle market.

---
## 1. Problem Statement

The used car market is growing rapidly, and determining the right selling price for a pre-owned vehicle is a challenge for both buyers and sellers. The goal of this project is to build a predictive model that can accurately estimate the selling price of a used car based on its characteristics — including its current showroom price, age, fuel type, transmission, and kilometers driven. By leveraging machine learning techniques, we aim to provide data-driven price estimates that can assist dealers and individuals in making informed pricing decisions.

---
## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('All libraries imported successfully.')

---
## 3. Load Dataset

We load the car dataset from the raw data directory. The dataset contains information about used cars including their selling price, present showroom price, and various other attributes.

In [ ]:
df = pd.read_csv('../data/raw/car_data.csv')
df.head(10)

In [ ]:
df.shape

---
## 4. Understand the Dataset

Let's explore the dataset to understand its structure, data types, distributions, and identify any potential issues.

### Dataset Shape

In [ ]:
print(f'Number of rows: {df.shape[0]}')
print(f'Number of columns: {df.shape[1]}')

The dataset has approximately 301 rows and 9 columns. Each row represents a used vehicle listing with its specifications and selling price.

### Column Names

In [ ]:
df.columns.tolist()

We have 9 features covering vehicle identification, pricing, usage, fuel type, seller information, transmission, and ownership history.

### Data Types

In [ ]:
df.dtypes

We have 3 categorical columns (Car_Name, Fuel_Type, Selling_type, Transmission) stored as objects, and the remaining are numerical. Selling_Price is our target variable stored as a float.

### Dataset Info

In [ ]:
df.info()

The info summary confirms the data types and shows non-null counts for each column. Memory usage is minimal, making this dataset efficient to work with.

### Descriptive Statistics

In [ ]:
df.describe()

Key observations: Selling prices range widely from very low values (likely two-wheelers) to over 35 Lakhs. The mean selling price is around 4.66 Lakhs, while the median is lower, indicating a right-skewed distribution. Driven kilometers also shows high variance.

### Missing Values

In [ ]:
df.isnull().sum()

There are no missing values in the dataset, which saves us significant preprocessing time. However, we should still check for empty rows or inconsistent entries.

### Duplicate Rows

In [ ]:
print(f'Number of duplicate rows: {df.duplicated().sum()}')

We have a small number of duplicate entries (if any). We will handle these during the data cleaning phase.

### Unique Values per Column

In [ ]:
df.nunique()

Car_Name has a high number of unique values, suggesting a wide variety of vehicle models in the dataset. Fuel_Type, Selling_type, and Transmission have limited categories, making them suitable for encoding.

---
## 5. Data Cleaning

We will clean the dataset by handling duplicates, checking for inconsistent values in categorical columns, fixing data type issues, and removing any empty rows.

In [ ]:
# Drop any completely empty rows
initial_shape = df.shape[0]
df = df.dropna()
print(f'Rows removed (empty/NaN): {initial_shape - df.shape[0]}')

In [ ]:
# Drop duplicate rows
duplicates_count = df.duplicated().sum()
df = df.drop_duplicates()
print(f'Duplicate rows removed: {duplicates_count}')
print(f'Dataset shape after removing duplicates: {df.shape}')

### Check Categorical Columns for Inconsistencies

In [ ]:
print('Fuel Type Distribution:')
print(df['Fuel_Type'].value_counts())
print()
print('Selling Type Distribution:')
print(df['Selling_type'].value_counts())
print()
print('Transmission Distribution:')
print(df['Transmission'].value_counts())
print()
print('Owner Distribution:')
print(df['Owner'].value_counts())

The categorical columns look clean with consistent values. Petrol vehicles dominate the dataset, most sales are through dealers, and manual transmission is far more common than automatic.

In [ ]:
# Reset index after cleaning
df = df.reset_index(drop=True)
print(f'Dataset shape after cleaning: {df.shape}')

The dataset is now clean and ready for exploratory analysis.

---
## 6. Exploratory Data Analysis

Let's analyze the data to find patterns and relationships that will inform our modeling decisions.

### Target Variable Analysis

In [ ]:
print('Selling Price Statistics:')
print(f'  Mean:   {df["Selling_Price"].mean():.2f} Lakhs')
print(f'  Median: {df["Selling_Price"].median():.2f} Lakhs')
print(f'  Min:    {df["Selling_Price"].min():.2f} Lakhs')
print(f'  Max:    {df["Selling_Price"].max():.2f} Lakhs')
print(f'  Std:    {df["Selling_Price"].std():.2f} Lakhs')
print(f'  Skew:   {df["Selling_Price"].skew():.2f}')

The selling price is positively skewed, meaning most vehicles sell at lower prices while a few high-end vehicles push the distribution to the right. The large gap between mean and median confirms this skewness.

### Correlation Analysis

In [ ]:
correlation_matrix = df.select_dtypes(include=[np.number]).corr()
selling_price_corr = correlation_matrix['Selling_Price'].sort_values(ascending=False)
print('Correlation with Selling Price:')
print(selling_price_corr)

Present_Price has the strongest positive correlation with Selling_Price, which makes intuitive sense — vehicles with higher showroom prices tend to retain higher resale values. Year also shows a positive correlation, indicating newer vehicles sell for more.

### Categorical Analysis

In [ ]:
print('=== Fuel Type ===')
print(df['Fuel_Type'].value_counts())
print(f'Percentage: \n{df["Fuel_Type"].value_counts(normalize=True).mul(100).round(1)}')
print()
print('=== Selling Type ===')
print(df['Selling_type'].value_counts())
print()
print('=== Transmission ===')
print(df['Transmission'].value_counts())
print()
print('=== Owner ===')
print(df['Owner'].value_counts())

Petrol vehicles make up the majority of listings, followed by diesel. CNG vehicles are rare. Most vehicles are sold through dealers and have manual transmission. The majority are first-owner vehicles.

### Numerical Analysis

In [ ]:
print('=== Year Statistics ===')
print(f'Range: {df["Year"].min()} to {df["Year"].max()}')
print(f'Most common year: {df["Year"].mode()[0]}')
print()
print('=== Present Price Statistics ===')
print(f'Range: {df["Present_Price"].min():.2f} to {df["Present_Price"].max():.2f} Lakhs')
print(f'Mean: {df["Present_Price"].mean():.2f} Lakhs')
print()
print('=== Driven Kilometers Statistics ===')
print(f'Range: {df["Driven_kms"].min()} to {df["Driven_kms"].max()} km')
print(f'Mean: {df["Driven_kms"].mean():.0f} km')

Vehicles span manufacturing years from 2003 to 2018. Present prices range widely, suggesting a mix of budget and premium vehicles. Driven kilometers also varies significantly, from barely used to heavily driven vehicles.

### Outlier Detection

In [ ]:
Q1 = df['Selling_Price'].quantile(0.25)
Q3 = df['Selling_Price'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Selling_Price'] < lower_bound) | (df['Selling_Price'] > upper_bound)]
print(f'Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}')
print(f'Lower Bound: {lower_bound:.2f}, Upper Bound: {upper_bound:.2f}')
print(f'Number of outliers in Selling_Price: {len(outliers)}')
print(f'Percentage of outliers: {len(outliers) / len(df) * 100:.1f}%')

There are some outliers in the selling price, representing high-value premium vehicles. We will keep these outliers as they represent genuine data points — premium cars naturally have higher resale values, and removing them would bias our model toward budget vehicles.

### Note on Dataset Composition

> **Important:** This dataset includes both **cars** and **two-wheelers** (motorcycles and scooters such as Royal Enfield Thunder 500, Bajaj Pulsar, Honda Activa, etc.). The presence of two-wheelers contributes to the wide price range and lower-end pricing distribution. This is worth noting as it affects the model's predictions across vehicle categories.

In [ ]:
# Check for two-wheeler brands in the dataset
two_wheeler_keywords = ['royal enfield', 'bajaj', 'hero', 'honda activa', 'tvs', 'yamaha', 'ktm', 'pulsar', 'activa']
two_wheelers = df[df['Car_Name'].str.lower().str.contains('|'.join(two_wheeler_keywords), na=False)]
print(f'Approximate number of two-wheeler entries: {len(two_wheelers)}')
print(f'Two-wheeler names found:')
print(two_wheelers['Car_Name'].unique())

---
## 7. Feature Engineering

We will create new features and encode categorical variables to prepare the data for machine learning models.

### Calculate Car Age

In [ ]:
current_year = 2025
df['Car_Age'] = current_year - df['Year']
print(f'Car Age range: {df["Car_Age"].min()} to {df["Car_Age"].max()} years')
print(f'Average Car Age: {df["Car_Age"].mean():.1f} years')

Car age is a more intuitive feature than the manufacturing year and directly represents depreciation over time.

### Extract Brand Name

In [ ]:
def extract_brand(car_name):
    brand = car_name.split(' ')[0].lower()
    return brand

df['Brand'] = df['Car_Name'].apply(extract_brand)
print(f'Number of unique brands: {df["Brand"].nunique()}')
print()
print('Brand distribution:')
print(df['Brand'].value_counts())

We extracted the brand name from the car name for analysis purposes. Maruti is the most common brand, followed by other popular Indian market brands.

### Encode Categorical Variables

In [ ]:
fuel_type_map = {'Petrol': 0, 'Diesel': 1, 'CNG': 2}
df['Fuel_Type_Encoded'] = df['Fuel_Type'].map(fuel_type_map)

selling_type_map = {'Dealer': 0, 'Individual': 1}
df['Selling_Type_Encoded'] = df['Selling_type'].map(selling_type_map)

transmission_map = {'Manual': 0, 'Automatic': 1}
df['Transmission_Encoded'] = df['Transmission'].map(transmission_map)

print('Encoding complete.')
print(f'Fuel Type mapping: {fuel_type_map}')
print(f'Selling Type mapping: {selling_type_map}')
print(f'Transmission mapping: {transmission_map}')

### Log Transformation of Target Variable

In [ ]:
print(f'Skewness of Selling_Price: {df["Selling_Price"].skew():.2f}')

df['Log_Selling_Price'] = np.log1p(df['Selling_Price'])
print(f'Skewness after log transform: {df["Log_Selling_Price"].skew():.2f}')

The log transformation reduces skewness significantly, making the distribution more normal. However, we will train our models on the original selling price for interpretability.

### Save Processed Data

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/car_data_processed.csv', index=False)
print('Processed data saved to ../data/processed/car_data_processed.csv')
print(f'Processed dataset shape: {df.shape}')
df.head()

---
## 8. Data Visualization

Let's visualize the data to better understand patterns, distributions, and relationships between features.

In [ ]:
os.makedirs('../reports/figures', exist_ok=True)

### 8.1 Selling Price Distribution

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['Selling_Price'], bins=30, kde=True, color='steelblue')
plt.title('Distribution of Selling Price', fontsize=16, fontweight='bold')
plt.xlabel('Selling Price (Lakhs)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../reports/figures/selling_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

The selling price distribution is right-skewed, with the majority of vehicles priced below 10 Lakhs. A long tail extends to higher-priced premium vehicles.

### 8.2 Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 8))
heatmap_cols = ['Selling_Price', 'Present_Price', 'Driven_kms', 'Car_Age',
                'Fuel_Type_Encoded', 'Selling_Type_Encoded', 'Transmission_Encoded', 'Owner']
sns.heatmap(df[heatmap_cols].corr(), annot=True, cmap='coolwarm', center=0,
            fmt='.2f', linewidths=0.5, square=True)
plt.title('Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

Present_Price has the strongest positive correlation with Selling_Price. Car_Age shows a negative correlation, confirming that older vehicles depreciate in value.

### 8.3 Fuel Type Distribution

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(x='Fuel_Type', data=df, palette='Set2', order=df['Fuel_Type'].value_counts().index)
plt.title('Distribution of Fuel Types', fontsize=16, fontweight='bold')
plt.xlabel('Fuel Type')
plt.ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
               ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/fuel_type_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

Petrol vehicles overwhelmingly dominate the dataset, followed by diesel. CNG vehicles are very rare in the used car market.

### 8.4 Transmission Distribution

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(x='Transmission', data=df, palette='Set2', order=df['Transmission'].value_counts().index)
plt.title('Distribution of Transmission Types', fontsize=16, fontweight='bold')
plt.xlabel('Transmission Type')
plt.ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
               ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/transmission_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

Manual transmission vehicles far outnumber automatic ones, reflecting the Indian market's historical preference for manual gearboxes.

### 8.5 Seller Type Comparison

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(x='Selling_type', data=df, palette='Set2', order=df['Selling_type'].value_counts().index)
plt.title('Distribution of Seller Types', fontsize=16, fontweight='bold')
plt.xlabel('Seller Type')
plt.ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
               ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/seller_type_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

Most vehicles in the dataset are sold through dealers rather than individual sellers, which may indicate a bias toward organized resale channels.

### 8.6 Present Price vs Selling Price

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Present_Price', y='Selling_Price', hue='Fuel_Type',
                data=df, palette='Set2', alpha=0.7, s=80)
plt.title('Present Price vs Selling Price', fontsize=16, fontweight='bold')
plt.xlabel('Present Price (Lakhs)')
plt.ylabel('Selling Price (Lakhs)')
plt.legend(title='Fuel Type')
plt.tight_layout()
plt.savefig('../reports/figures/present_vs_selling_price.png', dpi=150, bbox_inches='tight')
plt.show()

There is a strong positive linear relationship between present price and selling price. Diesel vehicles tend to cluster at higher price points, while petrol vehicles span a wider range.

### 8.7 Car Age vs Selling Price

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Car_Age', y='Selling_Price', hue='Fuel_Type',
                data=df, palette='Set2', alpha=0.7, s=80)
plt.title('Car Age vs Selling Price', fontsize=16, fontweight='bold')
plt.xlabel('Car Age (Years)')
plt.ylabel('Selling Price (Lakhs)')
plt.legend(title='Fuel Type')
plt.tight_layout()
plt.savefig('../reports/figures/car_age_vs_selling_price.png', dpi=150, bbox_inches='tight')
plt.show()

Newer vehicles (lower car age) tend to have higher selling prices, confirming the expected depreciation trend. The relationship is more pronounced for diesel vehicles.

### 8.8 Driven Kilometers vs Selling Price

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Driven_kms', y='Selling_Price', data=df, color='steelblue', alpha=0.6, s=60)
plt.title('Driven Kilometers vs Selling Price', fontsize=16, fontweight='bold')
plt.xlabel('Kilometers Driven')
plt.ylabel('Selling Price (Lakhs)')
plt.tight_layout()
plt.savefig('../reports/figures/driven_kms_vs_selling_price.png', dpi=150, bbox_inches='tight')
plt.show()

There is no strong linear relationship between kilometers driven and selling price, suggesting that other factors like brand, age, and present price play a more decisive role in pricing.

### 8.9 Selling Price by Fuel Type

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='Fuel_Type', y='Selling_Price', data=df, palette='Set2',
            order=['Petrol', 'Diesel', 'CNG'])
plt.title('Selling Price by Fuel Type', fontsize=16, fontweight='bold')
plt.xlabel('Fuel Type')
plt.ylabel('Selling Price (Lakhs)')
plt.tight_layout()
plt.savefig('../reports/figures/selling_price_by_fuel_type.png', dpi=150, bbox_inches='tight')
plt.show()

Diesel vehicles command significantly higher resale prices compared to petrol and CNG vehicles. This likely reflects the higher initial cost and better fuel economy of diesel engines.

### 8.10 Selling Price by Transmission

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='Transmission', y='Selling_Price', data=df, palette='Set2',
            order=['Manual', 'Automatic'])
plt.title('Selling Price by Transmission Type', fontsize=16, fontweight='bold')
plt.xlabel('Transmission')
plt.ylabel('Selling Price (Lakhs)')
plt.tight_layout()
plt.savefig('../reports/figures/selling_price_by_transmission.png', dpi=150, bbox_inches='tight')
plt.show()

Automatic transmission vehicles tend to have higher selling prices, reflecting their premium positioning in the market.

### 8.11 Top 10 Brands by Average Selling Price

In [ ]:
brand_avg_price = df.groupby('Brand')['Selling_Price'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
brand_avg_price.plot(kind='barh', color='steelblue', edgecolor='black', linewidth=0.5)
plt.title('Top 10 Brands by Average Selling Price', fontsize=16, fontweight='bold')
plt.xlabel('Average Selling Price (Lakhs)')
plt.ylabel('Brand')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/figures/top_brands_avg_price.png', dpi=150, bbox_inches='tight')
plt.show()

Premium and SUV-focused brands command the highest average resale prices. Budget-friendly brands like Maruti have lower average prices due to their range of economy vehicles.

### 8.12 Manufacturing Year vs Average Selling Price

In [ ]:
year_avg = df.groupby('Year')['Selling_Price'].mean()

plt.figure(figsize=(10, 6))
plt.plot(year_avg.index, year_avg.values, marker='o', color='coral', linewidth=2, markersize=8)
plt.fill_between(year_avg.index, year_avg.values, alpha=0.2, color='coral')
plt.title('Average Selling Price by Manufacturing Year', fontsize=16, fontweight='bold')
plt.xlabel('Manufacturing Year')
plt.ylabel('Average Selling Price (Lakhs)')
plt.xticks(year_avg.index, rotation=45)
plt.tight_layout()
plt.savefig('../reports/figures/year_vs_avg_selling_price.png', dpi=150, bbox_inches='tight')
plt.show()

There is a clear upward trend in average selling prices for newer manufacturing years, consistent with lower depreciation for recently manufactured vehicles.

---
## 9. Prepare Training Data

We select the most relevant features for our model and split the data into training and testing sets using an 80/20 split.

In [ ]:
feature_columns = ['Present_Price', 'Driven_kms', 'Car_Age',
                   'Fuel_Type_Encoded', 'Selling_Type_Encoded',
                   'Transmission_Encoded', 'Owner']

X = df[feature_columns]
y = df['Selling_Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training set size: {X_train.shape[0]} samples')
print(f'Testing set size:  {X_test.shape[0]} samples')
print(f'Number of features: {X_train.shape[1]}')
print(f'\nFeatures used: {feature_columns}')

---
## 10. Train Machine Learning Models

We will train four different regression models and compare their performance to identify the best predictor for used car prices.

In [ ]:
results = {}

### 10.1 Linear Regression

Linear Regression is our baseline model. It assumes a linear relationship between features and the target variable, making it simple but effective for understanding feature contributions.

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_predictions = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_predictions)
lr_mse = mean_squared_error(y_test, lr_predictions)
lr_rmse = np.sqrt(lr_mse)
lr_r2 = r2_score(y_test, lr_predictions)

results['Linear Regression'] = {
    'MAE': lr_mae, 'MSE': lr_mse, 'RMSE': lr_rmse, 'R2_Score': lr_r2
}

print('Linear Regression Results:')
print(f'  MAE:      {lr_mae:.4f}')
print(f'  MSE:      {lr_mse:.4f}')
print(f'  RMSE:     {lr_rmse:.4f}')
print(f'  R² Score: {lr_r2:.4f}')

### 10.2 Decision Tree Regressor

Decision Tree can capture non-linear relationships and interactions between features. It splits the data recursively based on feature thresholds to make predictions.

In [ ]:
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)
dt_predictions = dt_model.predict(X_test)

dt_mae = mean_absolute_error(y_test, dt_predictions)
dt_mse = mean_squared_error(y_test, dt_predictions)
dt_rmse = np.sqrt(dt_mse)
dt_r2 = r2_score(y_test, dt_predictions)

results['Decision Tree'] = {
    'MAE': dt_mae, 'MSE': dt_mse, 'RMSE': dt_rmse, 'R2_Score': dt_r2
}

print('Decision Tree Results:')
print(f'  MAE:      {dt_mae:.4f}')
print(f'  MSE:      {dt_mse:.4f}')
print(f'  RMSE:     {dt_rmse:.4f}')
print(f'  R² Score: {dt_r2:.4f}')

### 10.3 Random Forest Regressor

Random Forest is an ensemble of decision trees that reduces overfitting by averaging predictions from multiple trees trained on random subsets of data and features.

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_mse = mean_squared_error(y_test, rf_predictions)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_predictions)

results['Random Forest'] = {
    'MAE': rf_mae, 'MSE': rf_mse, 'RMSE': rf_rmse, 'R2_Score': rf_r2
}

print('Random Forest Results:')
print(f'  MAE:      {rf_mae:.4f}')
print(f'  MSE:      {rf_mse:.4f}')
print(f'  RMSE:     {rf_rmse:.4f}')
print(f'  R² Score: {rf_r2:.4f}')

### 10.4 Gradient Boosting Regressor

Gradient Boosting builds trees sequentially, where each new tree corrects the errors of the previous one. This often yields the best predictive performance among tree-based methods.

In [ ]:
gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)
gb_predictions = gb_model.predict(X_test)

gb_mae = mean_absolute_error(y_test, gb_predictions)
gb_mse = mean_squared_error(y_test, gb_predictions)
gb_rmse = np.sqrt(gb_mse)
gb_r2 = r2_score(y_test, gb_predictions)

results['Gradient Boosting'] = {
    'MAE': gb_mae, 'MSE': gb_mse, 'RMSE': gb_rmse, 'R2_Score': gb_r2
}

print('Gradient Boosting Results:')
print(f'  MAE:      {gb_mae:.4f}')
print(f'  MSE:      {gb_mse:.4f}')
print(f'  RMSE:     {gb_rmse:.4f}')
print(f'  R² Score: {gb_r2:.4f}')

---
## 11. Model Evaluation

Let's compare all four models side by side to determine the best performer.

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.round(4)
results_df = results_df.sort_values('R2_Score', ascending=False)
print('Model Comparison (sorted by R² Score):')
print('=' * 65)
print(results_df)
print('=' * 65)

best_model_name = results_df['R2_Score'].idxmax()
print(f'\nBest Model: {best_model_name} (R² = {results_df.loc[best_model_name, "R2_Score"]:.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = ['steelblue', 'coral', '#66c2a5', '#fc8d62']

# R² Score comparison
model_names = results_df.index.tolist()
r2_scores = results_df['R2_Score'].values
bars1 = axes[0].barh(model_names, r2_scores, color=colors[:len(model_names)], edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('R² Score')
axes[0].set_title('Model Comparison — R² Score', fontsize=14, fontweight='bold')
axes[0].set_xlim(0, 1)
for bar, score in zip(bars1, r2_scores):
    axes[0].text(score + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{score:.4f}', va='center', fontweight='bold')

# RMSE comparison
rmse_scores = results_df['RMSE'].values
bars2 = axes[1].barh(model_names, rmse_scores, color=colors[:len(model_names)], edgecolor='black', linewidth=0.5)
axes[1].set_xlabel('RMSE (Lakhs)')
axes[1].set_title('Model Comparison — RMSE', fontsize=14, fontweight='bold')
for bar, score in zip(bars2, rmse_scores):
    axes[1].text(score + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{score:.4f}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

The ensemble models (Random Forest and Gradient Boosting) outperform the simpler models significantly. The best model achieves a high R² score, indicating it explains most of the variance in selling prices. Linear Regression serves as a reasonable baseline but struggles with the non-linear relationships in the data.

---
## 12. Feature Importance Analysis

We use the Random Forest model to identify which features have the greatest influence on predicted selling prices.

In [ ]:
feature_importance = pd.Series(rf_model.feature_importances_, index=feature_columns)
feature_importance = feature_importance.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feature_importance.plot(kind='barh', color='steelblue', edgecolor='black', linewidth=0.5)
plt.title('Feature Importance (Random Forest)', fontsize=16, fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('../reports/figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nFeature Importance Ranking:')
for feature, importance in feature_importance.sort_values(ascending=False).items():
    print(f'  {feature:25s} {importance:.4f}')

**Present_Price** is by far the most important feature, followed by **Car_Age** and **Driven_kms**. This makes intuitive sense — the current showroom price is the strongest indicator of a vehicle's resale value, while age and usage determine the depreciation.

---
## 13. Predictions & Model Diagnostics

Let's evaluate the best model's predictions in detail using diagnostic visualizations and sample predictions.

In [ ]:
models = {
    'Linear Regression': lr_model,
    'Decision Tree': dt_model,
    'Random Forest': rf_model,
    'Gradient Boosting': gb_model
}

predictions_dict = {
    'Linear Regression': lr_predictions,
    'Decision Tree': dt_predictions,
    'Random Forest': rf_predictions,
    'Gradient Boosting': gb_predictions
}

best_model = models[best_model_name]
best_predictions = predictions_dict[best_model_name]

print(f'Using best model: {best_model_name}')

### Actual vs Predicted Values

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(y_test, best_predictions, alpha=0.6, color='steelblue', s=60, edgecolors='white', linewidth=0.5)
max_val = max(y_test.max(), best_predictions.max())
plt.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect Prediction')
plt.title(f'Actual vs Predicted Selling Price ({best_model_name})', fontsize=16, fontweight='bold')
plt.xlabel('Actual Selling Price (Lakhs)')
plt.ylabel('Predicted Selling Price (Lakhs)')
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig('../reports/figures/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

Points closely aligned with the red diagonal line indicate accurate predictions. The model performs well across the price range, with slight deviations at higher price points.

### Residual Plot

In [ ]:
residuals = y_test.values - best_predictions

plt.figure(figsize=(10, 6))
plt.scatter(best_predictions, residuals, alpha=0.6, color='coral', s=60, edgecolors='white', linewidth=0.5)
plt.axhline(y=0, color='black', linestyle='--', linewidth=2)
plt.title(f'Residual Plot ({best_model_name})', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Selling Price (Lakhs)')
plt.ylabel('Residuals (Lakhs)')
plt.tight_layout()
plt.savefig('../reports/figures/residual_plot.png', dpi=150, bbox_inches='tight')
plt.show()

The residuals are randomly distributed around zero with no clear pattern, indicating the model captures the underlying trends well without systematic bias.

### Sample Predictions

In [ ]:
comparison_df = pd.DataFrame({
    'Actual Price (Lakhs)': y_test.values[:10],
    'Predicted Price (Lakhs)': np.round(best_predictions[:10], 2),
    'Difference (Lakhs)': np.round(y_test.values[:10] - best_predictions[:10], 2)
})
comparison_df.index = range(1, 11)
comparison_df.index.name = 'Sample'
print(f'Sample Predictions — {best_model_name}')
print('=' * 60)
print(comparison_df.to_string())

### Save Best Model

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/trained_model.pkl')
print(f'Best model ({best_model_name}) saved to ../models/trained_model.pkl')

---
## 14. Business Insights

Based on our analysis and modeling, here are the key business insights for the used car market:

- **Present showroom price is the strongest predictor** of resale value. Vehicles with higher original prices retain better value in the used market, making brand positioning a critical factor.

- **Vehicle age drives depreciation significantly.** Each additional year of age reduces the selling price noticeably. Vehicles under 5 years old retain the best value, while those older than 10 years see steep depreciation.

- **Diesel vehicles command premium resale prices** compared to petrol and CNG counterparts. Buyers in the used market value the fuel efficiency and durability associated with diesel engines.

- **Automatic transmission adds value.** Despite being less common, automatic vehicles fetch higher prices, reflecting growing consumer preference for convenience.

- **Kilometers driven has a weaker impact than expected.** While high mileage does affect pricing, it is not as decisive as age or original price, suggesting buyers weigh overall condition and brand more heavily.

- **Dealer sales dominate the market.** The majority of transactions happen through dealers rather than individuals, indicating that dealer certification and trust play a role in buyer decisions.

- **The dataset includes two-wheelers alongside cars**, which broadens the price range and means our model generalizes across vehicle types — a useful feature for multi-category pricing platforms.

- **First-owner vehicles sell better.** Vehicles with zero or one previous owner have higher resale values, highlighting the importance of ownership history in pricing decisions.

---
## 15. Conclusion

### Project Summary

In this project, we built a complete machine learning pipeline to predict used car selling prices. Starting from raw data exploration and cleaning, we engineered meaningful features like car age and brand extraction, and trained four regression models — Linear Regression, Decision Tree, Random Forest, and Gradient Boosting.

### Key Results

- The **ensemble models** (Random Forest and Gradient Boosting) significantly outperformed simpler approaches.
- **Present Price**, **Car Age**, and **Driven Kilometers** emerged as the top three predictive features.
- The best model achieves a strong R² score, demonstrating reliable price prediction capability.

### Top Predictive Features
1. Present Price (showroom value)
2. Car Age (years since manufacture)
3. Driven Kilometers (usage)
4. Fuel Type
5. Transmission Type

### Future Improvements
- Incorporate additional features such as vehicle condition, service history, and location.
- Separate models for cars and two-wheelers could improve accuracy for each category.
- Hyperparameter tuning with GridSearchCV or RandomizedSearchCV could further optimize model performance.
- Deploying the model as a web application would make it accessible to end users.
- Expanding the dataset with more recent listings and additional brands would improve generalizability.

### Acknowledgment

This project was completed as part of the **CodeAlpha Data Science Internship — Task 3**. Thank you to **CodeAlpha** for providing this learning opportunity and fostering hands-on experience in real-world data science workflows.

---
*Notebook created with Python, scikit-learn, pandas, matplotlib, and seaborn.*